# Tracking-phase validation test harness

End-to-end reproduction of what happens each time `Engine._tracking_validation_tracking_epoch(epoch)` is called during `tracking_trainloop`, **without touching W&B**.

The three sub-routines exercised here are:

1. **StereoMIS** &mdash; `_stereomis_validation_epoch`: TAP metrics (`delta_avg`, `OA`, `AJ`) on every VAL sequence + two MP4s (`dense_video`, `gt_vs_pred_video`) on one random sequence per epoch.
2. **STIR** &mdash; `_stir_validation_epoch`: self-supervised scalars (`cycle_err_mean_px`, `mean_step_px`, `visibility_ratio`) + one `dense_video` on one random sequence per epoch.
3. **Pseudo novel-view** &mdash; `_tracking_validate_pseudo_novelview`: `mean_l2_px` on one synthesized window + `sparse_gt_vs_pred_video`.

A lightweight **wandb shim** is attached to `engine.wandb` so that every `log(...)` call (scalars *and* `wandb.Video`) is captured into a list instead of being sent anywhere. The generated MP4s are still written to `<RUN_DIR>/tracking_val/` and played back inline below.

## 0. Environment

In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from gatetracker.env_bootstrap import setdefault_cpu_thread_env, require_dotenv_before_pipeline

setdefault_cpu_thread_env()
require_dotenv_before_pipeline(purpose="validation-test-notebook")

%load_ext autoreload
%autoreload 2

import copy
import glob
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
import yaml
from dotmap import DotMap
from IPython.display import Video, display

torch.set_grad_enabled(False)
print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 1. Load `config/tracking.yaml`

Lightweight overrides keep the notebook cheap:

- `DISTRIBUTE=singlegpu` (no torchrun).
- `NO_WANDB=True` (we attach a local shim further down).
- `TRACKING_STEREOMIS_VAL_MAX_FRAMES=32` so every StereoMIS sequence is capped to 32 frames.
- `TRACKING_VAL_RENDER_VIDEOS_ALL_SEQS=True` so we actually get a video *for every* StereoMIS VAL sequence &mdash; the production code only renders one random seq/epoch.

Tune `OVERRIDES` below to match what you want to exercise.

In [ ]:
CONFIG_PATH = os.path.join(REPO_ROOT, "config", "tracking.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    raw = yaml.safe_load(f)
params = raw.get("parameters", raw)
cfg_dict = {
    k: (v["value"] if isinstance(v, dict) and "value" in v else v)
    for k, v in params.items()
}

OVERRIDES: Dict[str, Any] = {
    "DISTRIBUTE": "singlegpu",
    "NO_WANDB": True,
    "FEWFRAMES": True,
    "BATCH_SIZE": 1,
    "WORKERS": 0,
    "TRACKING_STEREOMIS_VAL_MAX_FRAMES": 32,
    "TRACKING_STEREOMIS_VAL_GRID": 8,
    "TRACKING_VAL_RENDER_VIDEOS_ALL_SEQS": True,
    "TRACKING_VAL_PSEUDO_MAX_POINTS": 32,
    "TRACKING_VAL_PSEUDO_GRID_SIZE": 6,
}
cfg_dict.update(OVERRIDES)
config = DotMap(cfg_dict)
config["DISTRIBUTE"] = str(config.get("DISTRIBUTE", "singlegpu"))
config["PHASE"] = "tracking"
print(f"PHASE={config.PHASE} | DISTRIBUTE={config.DISTRIBUTE} | IMAGE={config.IMAGE_HEIGHT}x{config.IMAGE_WIDTH}")

## 2. Build dataset + `Engine` (no W&B)

In [ ]:
from gatetracker.data import initialize_from_config
from gatetracker.engine import Engine

result = initialize_from_config(
    config,
    inference=True,
    verbose=True,
    dataset_phase="tracking",
)
dataset = result["dataset"]
config = result["config"]
print({k: (len(v) if hasattr(v, "__len__") else type(v).__name__) for k, v in dataset.items()})

In [ ]:
engine = Engine(
    model=config.get("RUN", ""),
    dataset=dataset,
    config=config,
    no_wandb=True,
)
print(f"RUN_DIR = {engine.RUN_DIR}")
print(f"device  = {engine.device}")
print(f"wandb   = {engine.wandb}  (expected None)")

## 3. Bring up the tracking phase

`_init_tracking_phase()` is what `tracking_trainloop` calls once at the start: it builds the `TemporalTracker` (including the refinement net + optimizer) and the `SequenceWindowDataset` for `{Training, Validation}` that pseudo-novel-view validation reads from.

In [ ]:
engine._init_tracking_phase()

refine_ckpt = os.path.join(engine.MODELS_DIR, "tracking_refinement_net.pt")
if os.path.isfile(refine_ckpt):
    from gatetracker.distributed_context import unwrap_model

    unwrap_model(engine.temporal_tracker.refinement_net).load_state_dict(
        torch.load(refine_ckpt, map_location=engine.device, weights_only=True)
    )
    print(f"Loaded refinement_net from {refine_ckpt}")
else:
    print("No refinement checkpoint found -> using freshly initialised refinement_net.")

engine.temporal_tracker.eval()
seq_val = engine._tracking_datasets.get("Validation")
print(f"_tracking_datasets['Validation'] = {type(seq_val).__name__} | len={len(seq_val) if seq_val is not None else None}")

## 4. Wandb shim

Everything `engine.wandb.log({...})` would send is captured into `WANDB_LOG` so we can inspect scalars and `wandb.Video` references directly. This is the **only** place the notebook diverges from the real training loop.

In [ ]:
WANDB_LOG: List[Dict[str, Any]] = []

class _WandbShim:
    def __init__(self, store: List[Dict[str, Any]]):
        self._store = store

    def log(self, payload: Dict[str, Any], *args, **kwargs) -> None:
        self._store.append(copy.copy(payload))

engine.wandb = _WandbShim(WANDB_LOG)
print(f"engine.wandb -> {engine.wandb!r}")

In [ ]:
def payloads_to_dataframe(payloads: List[Dict[str, Any]]) -> pd.DataFrame:
    """Flatten captured wandb.log(...) payloads into a tidy (key, value) DataFrame.

    `wandb.Video` entries are rendered as 'Video(<path>)' strings so the table
    stays human-readable; the raw objects remain in ``payloads``.
    """
    import wandb
    rows = []
    for idx, payload in enumerate(payloads):
        for key, value in payload.items():
            if isinstance(value, wandb.Video):
                rendered = f"Video({getattr(value, '_path', '<unknown>')})"
            elif isinstance(value, float):
                rendered = float(value)
            elif isinstance(value, (int, bool, str)):
                rendered = value
            else:
                rendered = repr(value)
            rows.append({"payload_idx": idx, "key": key, "value": rendered})
    return pd.DataFrame(rows)

def play_videos(glob_pattern: str, max_videos: int = 4) -> None:
    """Embed MP4s matching ``glob_pattern`` (absolute path) inline in the notebook."""
    paths = sorted(glob.glob(glob_pattern))[:max_videos]
    if not paths:
        print(f"(no videos matched {glob_pattern})")
        return
    for p in paths:
        print(p)
        display(Video(p, embed=True))

VAL_DIR = os.path.join(engine.RUN_DIR, "tracking_val")
EPOCH = 0  # bumped between sections so video filenames stay distinct
print(f"tracking_val dir: {VAL_DIR}")

## 5. StereoMIS validation (`_stereomis_validation_epoch`)

Per VAL sequence:

- **Metrics** via `compute_tap_metrics(pred_tracks, gt_tracks, gt_vis, pred_vis)`
  - $\delta_{\text{avg}}$: average over $\{1,2,4,8,16\}$ px of the fraction of visible GT tracks within threshold (position accuracy).
  - $\text{OA}$: occlusion accuracy of the predicted visibility head.
  - $\text{AJ}$: Jaccard between GT-visible&amp;correct and predicted-visible, averaged over thresholds.
- **Videos** (one random seq per epoch, or all when `TRACKING_VAL_RENDER_VIDEOS_ALL_SEQS=True`):
  - `dense_video`: $10{\times}10$ grid tracked through the sequence, coloured trails.
  - `gt_vs_pred_video`: predicted tracks vs GT tracks side-by-side, coloured by per-point pixel error $\lVert p_t^{\text{pred}} - p_t^{\text{gt}} \rVert_2$.

In [ ]:
EPOCH = 0
WANDB_LOG.clear()

engine._stereomis_validation_epoch(EPOCH)

df_stereomis = payloads_to_dataframe(WANDB_LOG)
scalar_rows = df_stereomis[~df_stereomis["value"].astype(str).str.startswith("Video(")]
print(f"Captured {len(WANDB_LOG)} wandb.log payloads ({len(df_stereomis)} keys total)")
scalar_rows.pivot_table(index="payload_idx", columns="key", values="value", aggfunc="first")

In [ ]:
play_videos(os.path.join(VAL_DIR, f"stereomis_*_grid_ep{EPOCH:04d}.mp4"), max_videos=2)

In [ ]:
play_videos(os.path.join(VAL_DIR, f"stereomis_*_cmp_ep{EPOCH:04d}.mp4"), max_videos=2)

## 6. STIR validation (`_stir_validation_epoch`)

STIR has no pixel-level GT tracks in our loader, so this routine emits **self-supervised proxies**:

- `cycle_err_mean_px` = $\mathbb{E}_q \lVert p^{\text{fwd}}_0(q) - p^{\text{rev}}_0(q) \rVert_2$, where the reverse pass retracks from $p^{\text{fwd}}_{T-1}$ on frames flipped along time.
- `mean_step_px` = $\mathbb{E}_{q,t}\,\lVert p_{t+1}(q) - p_t(q) \rVert_2$ &mdash; motion sanity check.
- `visibility_ratio` = $\mathbb{E}\,\sigma(\text{vis\_logits})$.

Only **one** random STIR sequence per epoch is rendered to `dense_video`.

In [ ]:
EPOCH = 1
WANDB_LOG.clear()

engine._stir_validation_epoch(EPOCH)

df_stir = payloads_to_dataframe(WANDB_LOG)
print(f"Captured {len(WANDB_LOG)} payloads")
df_stir

In [ ]:
play_videos(os.path.join(VAL_DIR, f"stir_*_ep{EPOCH:04d}.mp4"), max_videos=2)

## 7. Pseudo novel-view validation (`_tracking_validate_pseudo_novelview`)

Synthesises one novel-view clip from the **first frame** of validation window `TRACKING_VAL_PSEUDO_BATCH_INDEX` (see `config/tracking.yaml`), re-tracks it with the current refinement net, and reports

$$
\text{mean\_l2\_px}
= \frac{\sum_{b,q,t} m_{bqt}\,\lVert p^{\text{pred}}_{bqt} - p^{\text{gt}}_{bqt} \rVert_2}{\sum_{b,q,t} m_{bqt}}
$$

where $m \in \{0,1\}^{B\times Q\times T}$ is the *composite* supervision mask (points both `frame_valid` under RGB warp AND labelled visible by pseudo-GT generator), and then writes `sparse_gt_vs_pred_video` showing GT tracks and predictions on the synthesised frames.

In [ ]:
EPOCH = 2
WANDB_LOG.clear()

engine._tracking_validate_pseudo_novelview(EPOCH)

df_pseudo = payloads_to_dataframe(WANDB_LOG)
print(f"Captured {len(WANDB_LOG)} payloads")
df_pseudo

In [ ]:
play_videos(os.path.join(VAL_DIR, f"pseudo_novelview_ep{EPOCH:04d}.mp4"), max_videos=1)

## 8. End-to-end: `_tracking_validation_tracking_epoch`

Exactly what `tracking_trainloop` calls every epoch (on the main rank) after the Validation tracking pass finishes. Under DDP this is gated by `is_main_process()`; in this notebook `DISTRIBUTE=singlegpu` so it always runs.

In [ ]:
EPOCH = 3
WANDB_LOG.clear()

engine._tracking_validation_tracking_epoch(EPOCH)

df_all = payloads_to_dataframe(WANDB_LOG)
print(f"Captured {len(WANDB_LOG)} payloads / {len(df_all)} keys in total")
df_all.groupby(df_all["key"].str.split("/").str[:4].str.join("/")).size().rename("n_keys").to_frame()

In [ ]:
videos_in_payloads = [
    (idx, key, value)
    for idx, payload in enumerate(WANDB_LOG)
    for key, value in payload.items()
    if type(value).__name__ == "Video"
]
print(f"{len(videos_in_payloads)} wandb.Video entries referenced this epoch:")
for idx, key, value in videos_in_payloads:
    path = getattr(value, "_path", "<unknown>")
    print(f"  [{idx}] {key} -> {path}  (exists={os.path.isfile(path)})")

In [ ]:
play_videos(os.path.join(VAL_DIR, f"*ep{EPOCH:04d}.mp4"), max_videos=6)

## 9. All artifacts written to disk

Every MP4 the epoch would have uploaded to W&B is still on the filesystem under `<RUN_DIR>/tracking_val/`.

In [ ]:
val_files = sorted(Path(VAL_DIR).glob("*.mp4"))
pd.DataFrame(
    [
        {"file": p.name, "size_MB": round(p.stat().st_size / 2**20, 2)}
        for p in val_files
    ]
)